# Equilibrio entre Sectores: Gas y Electricidad

## Contexto

Este notebook resuelve un problema de **equilibrio entre dos sectores acoplados**: el sector del gas natural y el sector eléctrico. El acoplamiento ocurre porque el gas se usa como insumo para generar electricidad.

El problema se formula como un **MPEC (Mathematical Program with Equilibrium Constraints)**, que combina:
1.  Las restricciones primales de cada sector.
2.  Las restricciones duales (condiciones de KKT).
3.  Las condiciones de dualidad fuerte.

El objetivo es encontrar los **precios de equilibrio** ($\lambda_g$ para el gas, $\lambda_e$ para la electricidad) y las **cantidades producidas** ($g_1, g_2, g_e, p$).

## Formulación Matemática

### Sector Gas (PG)

$$
\min_{g_1, g_2} \quad g_1 + 3g_2
$$

Sujeto a:
*   $g_1 + g_2 = 12 + g_e \quad : \lambda_g$ (balance de gas)
*   $g_1 \le 10 \quad : \mu_1$ (capacidad productor 1)
*   $g_2 \le 14 \quad : \mu_2$ (capacidad productor 2)
*   $g_1, g_2 \ge 0$

### Sector Eléctrico (PE)

$$
\min_{g_e, p} \quad \lambda_g g_e + 2p
$$

Sujeto a:
*   $\frac{1}{2} g_e + p = 14 \quad : \lambda_e$ (balance eléctrico)
*   $p \le 10 \quad : \eta$ (capacidad planta eléctrica)
*   $p \ge 0$

### Condiciones de KKT (Duales)

**Dual de PG:**
$$
\max_{\lambda_g, \mu_1, \mu_2} \quad (12 + g_e)\lambda_g + 10\mu_1 + 14\mu_2
$$
Sujeto a:
*   $\lambda_g + \mu_1 \le 1$
*   $\lambda_g + \mu_2 \le 3$
*   $\mu_1, \mu_2 \ge 0$, $\lambda_g$ libre.

**Dual de PE:**
$$
\max_{\lambda_e, \eta} \quad 14\lambda_e + 10\eta
$$
Sujeto a:
*   $\frac{1}{2}\lambda_e \le \lambda_g$
*   $\lambda_e + \eta \le 2$
*   $\eta \ge 0$, $\lambda_e$ libre.

### Dualidad Fuerte

Para cada sector, el valor óptimo del primal debe igualar al del dual:

**PG:** $g_1 + 3g_2 = (12 + g_e)\lambda_g + 10\mu_1 + 14\mu_2$

**PE:** $\lambda_g g_e + 2p = 14\lambda_e + 10\eta$

In [1]:
import pyomo.environ as pyo
import pandas as pd

## Implementación en Pyomo

Vamos a construir el MPEC apilando todas las restricciones en un solo modelo. Como el problema tiene términos bilineales (productos de variables), usaremos el solver **SCIP** (global, open source) para encontrar la solución.

In [13]:
# --- CELDA 2: Modelo MPEC para el Equilibrio entre Sectores (con cotas) ---
import pyomo.environ as pyo

m = pyo.ConcreteModel(name="Equilibrium_Sectors")

# --- Variables ---
# Primales
m.g1 = pyo.Var(domain=pyo.NonNegativeReals)
m.g2 = pyo.Var(domain=pyo.NonNegativeReals)
m.ge = pyo.Var(domain=pyo.NonNegativeReals)
m.p  = pyo.Var(domain=pyo.NonNegativeReals)

# Duales de PG
m.lambda_g = pyo.Var(domain=pyo.Reals)  # Libre
m.mu1 = pyo.Var(domain=pyo.NonNegativeReals)
m.mu2 = pyo.Var(domain=pyo.NonNegativeReals)

# Duales de PE
m.lambda_e = pyo.Var(domain=pyo.Reals)  # Libre
m.eta = pyo.Var(domain=pyo.NonNegativeReals)

# --- Restricciones Primales PG ---
m.pg_balance = pyo.Constraint(expr=m.g1 + m.g2 == 12 + m.ge)
m.pg_cap1 = pyo.Constraint(expr=m.g1 <= 10)
m.pg_cap2 = pyo.Constraint(expr=m.g2 <= 14)

# --- Restricciones Primales PE ---
m.pe_balance = pyo.Constraint(expr=0.5*m.ge + m.p == 14)
m.pe_cap = pyo.Constraint(expr=m.p <= 10)

# --- Restricciones Duales PG ---
m.dual_pg_1 = pyo.Constraint(expr=m.lambda_g + m.mu1 <= 1)
m.dual_pg_2 = pyo.Constraint(expr=m.lambda_g + m.mu2 <= 3)

# --- Restricciones Duales PE ---
m.dual_pe_ge = pyo.Constraint(expr=0.5*m.lambda_e <= m.lambda_g)
m.dual_pe_p  = pyo.Constraint(expr=m.lambda_e + m.eta <= 2)

# --- Dualidad Fuerte PG ---
m.strong_dual_pg = pyo.Constraint(
    expr=m.g1 + 3*m.g2 == (12 + m.ge)*m.lambda_g + 10*m.mu1 + 14*m.mu2
)

# --- Dualidad Fuerte PE ---
m.strong_dual_pe = pyo.Constraint(
    expr=m.lambda_g*m.ge + 2*m.p == 14*m.lambda_e + 10*m.eta
)

# --- Objetivo (Factibilidad) ---
m.obj = pyo.Objective(expr=0, sense=pyo.minimize)

# --- Cotas "salvajes" para los precios (guían al solver) ---
m.lambda_g.setlb(0)
m.lambda_g.setub(10)
m.lambda_e.setlb(0)
m.lambda_e.setub(10)

# --- Cotas para las variables primales ---
m.g1.setub(15)
m.g2.setub(20)
m.ge.setub(15)
m.p.setub(15)

# --- Resolver con SCIP (solver global) ---
solver = pyo.SolverFactory('scip_direct')
results = solver.solve(m)

print(f"Estado: {results.solver.termination_condition}")

Estado: optimal


## Resultados del Equilibrio

El solver encuentra los precios y cantidades de equilibrio. Verificamos que se cumplan las condiciones de dualidad fuerte para ambos sectores.

In [14]:
# --- CELDA 3: Mostrar Resultados del Equilibrio ---
print("\n--- SOLUCIÓN DEL EQUILIBRIO ENTRE SECTORES ---")
print(f"g1 (Producción Gas 1) = {pyo.value(m.g1):.4f}")
print(f"g2 (Producción Gas 2) = {pyo.value(m.g2):.4f}")
print(f"ge (Gas exportado a Electricidad) = {pyo.value(m.ge):.4f}")
print(f"p  (Producción Eléctrica) = {pyo.value(m.p):.4f}")
print("-" * 50)
print(f"lambda_g (Precio del Gas) = ${pyo.value(m.lambda_g):.4f}/unidad")
print(f"lambda_e (Precio de la Electricidad) = ${pyo.value(m.lambda_e):.4f}/unidad")
print("-" * 50)
print(f"mu1 (Dual capacidad Gas 1) = {pyo.value(m.mu1):.4f}")
print(f"mu2 (Dual capacidad Gas 2) = {pyo.value(m.mu2):.4f}")
print(f"eta (Dual capacidad Eléctrica) = {pyo.value(m.eta):.4f}")

# --- Verificación de Dualidad Fuerte ---
print("\n--- VERIFICACIÓN DE DUALIDAD FUERTE ---")
# PG: primal vs dual
primal_pg = pyo.value(m.g1) + 3*pyo.value(m.g2)
dual_pg = (12 + pyo.value(m.ge))*pyo.value(m.lambda_g) + 10*pyo.value(m.mu1) + 14*pyo.value(m.mu2)
print(f"Primal PG: {primal_pg:.4f} | Dual PG: {dual_pg:.4f} | Diferencia: {abs(primal_pg - dual_pg):.6f}")

# PE: primal vs dual
primal_pe = pyo.value(m.lambda_g)*pyo.value(m.ge) + 2*pyo.value(m.p)
dual_pe = 14*pyo.value(m.lambda_e) + 10*pyo.value(m.eta)
print(f"Primal PE: {primal_pe:.4f} | Dual PE: {dual_pe:.4f} | Diferencia: {abs(primal_pe - dual_pe):.6f}")


--- SOLUCIÓN DEL EQUILIBRIO ENTRE SECTORES ---
g1 (Producción Gas 1) = 10.0000
g2 (Producción Gas 2) = 14.0000
ge (Gas exportado a Electricidad) = 12.0000
p  (Producción Eléctrica) = 8.0000
--------------------------------------------------
lambda_g (Precio del Gas) = $1.0000/unidad
lambda_e (Precio de la Electricidad) = $2.0000/unidad
--------------------------------------------------
mu1 (Dual capacidad Gas 1) = 0.0000
mu2 (Dual capacidad Gas 2) = 2.0000
eta (Dual capacidad Eléctrica) = 0.0000

--- VERIFICACIÓN DE DUALIDAD FUERTE ---
Primal PG: 52.0000 | Dual PG: 52.0000 | Diferencia: 0.000000
Primal PE: 28.0000 | Dual PE: 28.0000 | Diferencia: 0.000000


prueba de valirdad unos valores de referencia

In [12]:
# --- CELDA 4: Verificar la solución de Pozo (fijando precios) ---
import pyomo.environ as pyo

# Creamos un modelo nuevo (copia del anterior)
m_verif = pyo.ConcreteModel(name="Verificacion_Pozo")

# Variables
m_verif.g1 = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.g2 = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.ge = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.p  = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.lambda_g = pyo.Var(domain=pyo.Reals)
m_verif.mu1 = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.mu2 = pyo.Var(domain=pyo.NonNegativeReals)
m_verif.lambda_e = pyo.Var(domain=pyo.Reals)
m_verif.eta = pyo.Var(domain=pyo.NonNegativeReals)

# Restricciones primales PG
m_verif.pg_balance = pyo.Constraint(expr=m_verif.g1 + m_verif.g2 == 12 + m_verif.ge)
m_verif.pg_cap1 = pyo.Constraint(expr=m_verif.g1 <= 10)
m_verif.pg_cap2 = pyo.Constraint(expr=m_verif.g2 <= 14)

# Restricciones primales PE
m_verif.pe_balance = pyo.Constraint(expr=0.5*m_verif.ge + m_verif.p == 14)
m_verif.pe_cap = pyo.Constraint(expr=m_verif.p <= 10)

# Restricciones duales PG
m_verif.dual_pg_1 = pyo.Constraint(expr=m_verif.lambda_g + m_verif.mu1 <= 1)
m_verif.dual_pg_2 = pyo.Constraint(expr=m_verif.lambda_g + m_verif.mu2 <= 3)

# Restricciones duales PE
m_verif.dual_pe_ge = pyo.Constraint(expr=0.5*m_verif.lambda_e <= m_verif.lambda_g)
m_verif.dual_pe_p  = pyo.Constraint(expr=m_verif.lambda_e + m_verif.eta <= 2)

# Dualidad fuerte PG
m_verif.strong_dual_pg = pyo.Constraint(
    expr=m_verif.g1 + 3*m_verif.g2 == (12 + m_verif.ge)*m_verif.lambda_g + 10*m_verif.mu1 + 14*m_verif.mu2
)

# Dualidad fuerte PE
m_verif.strong_dual_pe = pyo.Constraint(
    expr=m_verif.lambda_g*m_verif.ge + 2*m_verif.p == 14*m_verif.lambda_e + 10*m_verif.eta
)

# Objetivo (factibilidad)
m_verif.obj = pyo.Objective(expr=0, sense=pyo.minimize)

# --- FIJAMOS LOS PRECIOS DE POZO ---
m_verif.lambda_g.fix(3.0)
m_verif.lambda_e.fix(6.0)

# --- Valores iniciales (cerca de la solución de Pozo) ---
m_verif.g1.set_value(10.0)
m_verif.g2.set_value(10.0)
m_verif.ge.set_value(8.0)
m_verif.p.set_value(10.0)
m_verif.mu1.set_value(0.0)
m_verif.mu2.set_value(0.0)
m_verif.eta.set_value(0.0)

# --- Resolver con SCIP ---
solver = pyo.SolverFactory('scip_direct')
results = solver.solve(m_verif)

print("--- VERIFICACIÓN DE LA SOLUCIÓN DE POZO ---")
print(f"Estado: {results.solver.termination_condition}")

if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    print("✅ La solución de Pozo es FACTIBLE.")
    print(f"g1 = {pyo.value(m_verif.g1):.4f}")
    print(f"g2 = {pyo.value(m_verif.g2):.4f}")
    print(f"ge = {pyo.value(m_verif.ge):.4f}")
    print(f"p  = {pyo.value(m_verif.p):.4f}")
    print(f"mu1 = {pyo.value(m_verif.mu1):.4f}")
    print(f"mu2 = {pyo.value(m_verif.mu2):.4f}")
    print(f"eta = {pyo.value(m_verif.eta):.4f}")
else:
    print("❌ La solución de Pozo NO es factible con los precios fijados.")

NoFeasibleSolutionError: A feasible solution was not found, so no solution can be loaded. Please set opt.config.load_solutions=False and check results.solution_status and results.incumbent_objective before loading a solution.